# Baseline 1: expression-only model (MOLI with a single expression encoder)

One model per drug: expression (697 genes) -> encoder (Linear, BatchNorm, ReLU, Dropout) -> linear head predicting ln IC50, trained with MSE plus MOLI's triplet loss (sensitive/resistant = below/above the training median for that drug).
No mutation, copy number, methylation or drug structure, and **no sex information**. This is the floor the fusion models are compared against.

Data: everything comes from `DeepCDR.zip` (CCLE omics, GDSC ln IC50, drug SMILES/graphs). Target: **ln IC50** (regression). All three baselines use the same cell lines, drugs, split and evaluation code, so differences come from the model only.
Upload the zips to Drive and edit the paths in the first cell. Run with `SMOKE = True` first to check nothing crashes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DEEPCDR_ZIP = '/content/drive/MyDrive/DeepCDR.zip'   # edit
MOLI_ZIP    = '/content/drive/MyDrive/MOLI.zip'      # edit (for utils.AllTripletSelector)
OUT_DIR     = '/content/drive/MyDrive/sex_drug_response'   # shared by all 3 baselines: splits + predictions live here
DEPMAP_EXPR = None   # optional: path to DepMap OmicsExpressionProteinCodingGenesTPMLogp1.csv, to infer sex from XIST / Y genes

import os, sys, subprocess
os.makedirs(OUT_DIR, exist_ok=True)
def sh(cmd): return subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True).stdout

sh(f'unzip -q -o "{DEEPCDR_ZIP}" -d /content')
sh(f'unzip -q -o "{MOLI_ZIP}" -d /content')
sys.path.insert(0, '/content/MOLI')   # utils.py
DATA = '/content/DeepCDR/data'

## 1. Settings, data, sex and tissue

In [ ]:
import re, csv, json, copy, random
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split

SEEDS        = [42]            # training seeds; add more (e.g. [0, 1, 2]) for mean +- std
SPLIT_SEED   = 42              # fixed: the split must be identical in all three baselines
N_DRUGS      = 20              # best-covered drugs (with <=100 heavy atoms, so DeepCDR graphs fit)
SPLIT_MODE   = 'cell_line'     # 'cell_line': unseen cell lines | 'tissue': unseen tissues (HOLDOUT_TISSUES)
HOLDOUT_TISSUES = ['skin', 'large_intestine', 'pancreas', 'stomach']
VAL_FRAC, TEST_FRAC = 0.10, 0.20
SEX_SOURCE   = 'annotation'    # 'annotation' (CCLE Gender) or 'expression' (inferred; needs DEPMAP_EXPR)
MIN_PER_SEX  = 3               # min lines of EACH sex in a tissue for within-tissue comparisons
N_BOOT       = 1000
SMOKE        = False           # True: 3 drugs, 2 epochs, 100 bootstrap reps -> just checks nothing crashes
SEX_SPECIFIC = {'breast', 'ovary', 'prostate', 'endometrium', 'cervix', 'testis', 'vulva', 'fallopian_tube'}
Y_GENES      = {'RPS4Y1', 'DDX3Y', 'KDM5D', 'UTY', 'EIF1AY', 'USP9Y', 'ZFY'}
if SMOKE: N_DRUGS, N_BOOT = 3, 100
random.seed(0); np.random.seed(0)

In [ ]:
ic50 = pd.read_csv(f'{DATA}/CCLE/GDSC_IC50.csv', index_col=0)                      # rows 'GDSC:<id>' x cell lines (ln IC50)
rows = list(csv.reader(open(f'{DATA}/GDSC/1.Drug_listMon Jun 24 09_00_55 2019.csv')))
drugid2cid = {r[0]: r[5] for r in rows if r[5].isdigit()}                           # GDSC drug id -> PubChem CID

E  = pd.read_csv(f'{DATA}/CCLE/genomic_expression_561celllines_697genes_demap_features.csv', index_col=0)
C  = pd.read_csv(f'{DATA}/CCLE/genomic_copynumber_561celllines_710genes_demap_features.csv', index_col=0).fillna(0)
Me = pd.read_csv(f'{DATA}/CCLE/genomic_methylation_561celllines_808genes_demap_features.csv', index_col=0)
M  = pd.read_csv(f'{DATA}/CCLE/genomic_mutation_34673_demap_features.csv', index_col=0)   # 961 lines; we keep the 561

ann = pd.read_csv(f'{DATA}/CCLE/Cell_lines_annotations_20181226.txt', sep='\t').drop_duplicates('depMapID').set_index('depMapID')
all_lines = [l for l in E.index if l in set(ic50.columns) and l in set(M.index) and l in set(C.index) and l in set(Me.index)]
meta = pd.DataFrame({'sex_annot': ann['Gender'].where(ann['Gender'].isin(['male', 'female']), 'unknown').reindex(all_lines),
                     'tissue': ann['Site_Primary'].fillna('unknown').reindex(all_lines)}, index=all_lines)
meta['sex_annot'] = meta['sex_annot'].fillna('unknown'); meta['tissue'] = meta['tissue'].fillna('unknown')
print(len(all_lines), 'cell lines with all four omics + response')

# ---- optional: sex inferred from expression (XIST and Y-linked genes). Heuristic thresholds on log2(TPM+1): CHECK them.
def infer_sex_from_expression(path, lines):
    df = pd.read_csv(path)
    if 'IsDefaultEntryForModel' in df.columns: df = df[df['IsDefaultEntryForModel'] == 'Yes']
    df = df.set_index('ModelID' if 'ModelID' in df.columns else df.columns[0])
    df = df[~df.index.duplicated()].reindex(lines)
    sym = {c: c.split(' ')[0] for c in df.columns}
    ym = df[[c for c, s in sym.items() if s in Y_GENES]].mean(axis=1)
    xi = df[[c for c, s in sym.items() if s == 'XIST']].mean(axis=1)
    out = pd.Series(np.where(ym > 1.0, 'male', np.where(xi > 1.0, 'female', 'unknown')), index=lines)
    out[ym.isna().values] = 'unknown'
    return out

meta['sex_expr'] = infer_sex_from_expression(DEPMAP_EXPR, all_lines) if DEPMAP_EXPR else 'NA'
if DEPMAP_EXPR:
    print('annotation (rows) vs expression-inferred sex (columns):'); print(pd.crosstab(meta.sex_annot, meta.sex_expr))
    both = meta[meta.sex_annot.isin(['male', 'female']) & meta.sex_expr.isin(['male', 'female'])]
    mis = (both.sex_annot != both.sex_expr).groupby(both.tissue).agg(['sum', 'size']).rename(columns={'sum': 'mismatched', 'size': 'n'})
    print('\nannotation vs inferred sex, mismatches by tissue:'); print(mis[mis.mismatched > 0].sort_values('mismatched', ascending=False))
meta['sex'] = np.where((SEX_SOURCE == 'expression') & meta.sex_expr.isin(['male', 'female']), meta.sex_expr, meta.sex_annot)

# ---- drugs: PubChem CID, heavy-atom count from the SMILES file (DeepCDR pads graphs to 100 atoms)
smiles = {l.split('\t')[0]: l.split('\t')[1].strip() for l in open(f'{DATA}/223drugs_pubchem_smiles.txt') if '\t' in l}
n_heavy = {c: len(re.findall(r'\[[^\]]+\]|Br|Cl|[BCNOPSFI]|[bcnops]', s)) for c, s in smiles.items()}

long = ic50.loc[[i for i in ic50.index if i.split(':')[1] in drugid2cid], all_lines].copy()
long['cid'] = [drugid2cid[i.split(':')[1]] for i in long.index]
long = long.groupby('cid').mean().stack().dropna().rename('y').reset_index().rename(columns={'level_1': 'line'})   # dropna: newer pandas keeps missing cells
long = long[long['cid'].map(lambda c: n_heavy.get(c, 999) <= 100)]
print(len(long), 'line-drug pairs;', long.cid.nunique(), 'drugs with a graph of <=100 atoms')

## 2. Shared split (identical in all three baselines)
Cell lines are split once, stratified on **tissue x sex** (rare strata merged), and the split is saved to `OUT_DIR` as JSON. The first notebook you run creates it; the other two load it, together with the drug list, so all three models are scored on exactly the same test cell lines and drugs.
`SPLIT_MODE = 'tissue'` holds out whole tissues instead (unseen-tissue test).

In [ ]:
SPLIT_PATH = f"{OUT_DIR}/splits_{SPLIT_MODE}_{N_DRUGS}drugs.json"
tissue_of, sex_of = meta['tissue'].to_dict(), meta['sex'].to_dict()

def make_strata(lines, min_size=4):
    s = pd.Series({l: f"{tissue_of[l]}|{sex_of[l]}" for l in lines})
    sx = pd.Series({l: sex_of[l] for l in lines})
    small = s.map(s.value_counts()) < min_size
    s[small] = 'rare|' + sx[small]
    small = s.map(s.value_counts()) < min_size
    s[small] = 'rare'
    if (s == 'rare').sum() < 2: s[s == 'rare'] = s.value_counts().index[0]
    return s

if os.path.exists(SPLIT_PATH):
    sp = json.load(open(SPLIT_PATH)); print('Loaded existing split:', SPLIT_PATH)
    drugs, tr_lines, va_lines, te_lines = sp['drugs'], sp['train'], sp['val'], sp['test']
else:
    cov = long.groupby('cid')['line'].nunique().sort_values(ascending=False)
    drugs = list(cov.index[:N_DRUGS])
    lines = list(all_lines)
    if SPLIT_MODE == 'tissue':
        te_lines = [l for l in lines if tissue_of[l] in HOLDOUT_TISSUES]
        rest = [l for l in lines if l not in set(te_lines)]
    else:
        rest, te_lines = train_test_split(lines, test_size=TEST_FRAC, stratify=make_strata(lines).loc[lines].values, random_state=SPLIT_SEED)
    tr_lines, va_lines = train_test_split(rest, test_size=VAL_FRAC / (1 - TEST_FRAC) if SPLIT_MODE == 'cell_line' else VAL_FRAC,
                                          stratify=make_strata(rest).loc[rest].values, random_state=SPLIT_SEED)
    json.dump(dict(drugs=drugs, train=list(tr_lines), val=list(va_lines), test=list(te_lines), mode=SPLIT_MODE, seed=SPLIT_SEED), open(SPLIT_PATH, 'w'))
    print('Created and saved split:', SPLIT_PATH)

split_of = {**{l: 'train' for l in tr_lines}, **{l: 'val' for l in va_lines}, **{l: 'test' for l in te_lines}}
meta['split'] = meta.index.map(split_of)
assert not (set(tr_lines) & set(te_lines)) and not (set(va_lines) & set(te_lines)), 'train/test overlap!'

pairs = long[long['cid'].isin(drugs)].copy()
pairs = pairs.join(meta[['sex', 'sex_annot', 'sex_expr', 'tissue', 'split']], on='line').reset_index(drop=True)
lines_ordered = list(meta.index); line2i = {l: i for i, l in enumerate(lines_ordered)}
pairs['li'] = pairs['line'].map(line2i); pairs['di'] = pairs['cid'].map({c: i for i, c in enumerate(drugs)})
tr_idx = [line2i[l] for l in tr_lines]

# ---- data audit (proposal: are the panels balanced by sex? is sex missing?)
print(f"\n{len(drugs)} drugs | pairs per split: {pairs.groupby('split').size().to_dict()}")
print('lines per split x sex:'); print(meta.groupby(['split', 'sex']).size().unstack(fill_value=0))
ns = meta[~meta.tissue.isin(SEX_SPECIFIC)]
fs = ns.groupby('tissue')['sex'].agg(n='size', female=lambda s: (s == 'female').mean(), unknown=lambda s: (s == 'unknown').mean())
print('\nfemale share by tissue (non-sex-specific tissues with >=10 lines):'); print(fs[fs.n >= 10].sort_values('female').round(2))
print(f"\nlines with missing sex annotation: {(meta.sex_annot == 'unknown').sum()} of {len(meta)}")

A = {'E': E.loc[lines_ordered].values.astype(np.float32), 'C': C.loc[lines_ordered].values.astype(np.float32),
     'Me': Me.loc[lines_ordered].values.astype(np.float32), 'M': M.loc[lines_ordered].values.astype(np.float32)}

## 3. Evaluation functions (identical in all three baselines)
Metrics: **RMSE, Pearson, Spearman** on held-out ln IC50. Reported overall and per sex, plus the **female-male gap** (95% bootstrap CI over cell lines) in three views: all lines, non-sex-specific tissues only, and tissue-matched (average of within-tissue gaps). A permutation test shuffles sex labels within tissue.
Pooled Pearson/Spearman across drugs is inflated by between-drug differences in mean ln IC50, so the per-drug means are shown too.

In [ ]:
METRICS = ['RMSE', 'Pearson', 'Spearman']

def _stats(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    if len(y) < 3 or y.std() == 0 or p.std() == 0: return np.full(3, np.nan)
    return np.array([np.sqrt(np.mean((p - y) ** 2)), np.corrcoef(p, y)[0, 1], spearmanr(p, y)[0]])

def perf_table(df):
    """RMSE / Pearson / Spearman pooled over pairs, plus the mean of per-drug Pearson / Spearman."""
    out = []
    for grp, d in [('all', df), ('female', df[df.sex == 'female']), ('male', df[df.sex == 'male']), ('unknown sex', df[df.sex == 'unknown'])]:
        if len(d) == 0: continue
        r = dict(zip(METRICS, _stats(d.y, d.pred)))
        pdrug = np.array([_stats(g.y, g.pred) for _, g in d.groupby('cid') if len(g) >= 5])
        r['per-drug Pearson'] = np.nanmean(pdrug[:, 1]) if len(pdrug) else np.nan
        r['per-drug Spearman'] = np.nanmean(pdrug[:, 2]) if len(pdrug) else np.nan
        out.append({'group': grp, 'n_lines': d.line.nunique(), 'n_pairs': len(d), **r})
    return pd.DataFrame(out)

def gap_analysis(df, n_boot=N_BOOT, min_per_sex=MIN_PER_SEX, seed=0):
    """Female minus male gap with a cell-line bootstrap CI, in three views:
       all lines | non-sex-specific tissues | tissue-matched (average of within-tissue gaps)."""
    d = df[df.sex.isin(['female', 'male'])]
    if d.sex.nunique() < 2: print('need both sexes'); return None
    by_line = {l: (g.y.values, g.pred.values) for l, g in d.groupby('line')}
    meta_t = d.drop_duplicates('line').set_index('line')[['sex', 'tissue']]
    pooled = lambda ls: _stats(np.concatenate([by_line[l][0] for l in ls]), np.concatenate([by_line[l][1] for l in ls]))
    Fall, Mall = meta_t.index[meta_t.sex == 'female'].values, meta_t.index[meta_t.sex == 'male'].values
    ns = ~meta_t.tissue.isin(SEX_SPECIFIC)
    Fns, Mns = meta_t.index[(meta_t.sex == 'female') & ns].values, meta_t.index[(meta_t.sex == 'male') & ns].values
    tis_ok = [t for t, g in meta_t.groupby('tissue') if (g.sex == 'female').sum() >= min_per_sex and (g.sex == 'male').sum() >= min_per_sex]

    def simple(Fl, Ml):
        def f(rs):
            a, b = (Fl, Ml) if rs is None else (rs.choice(Fl, len(Fl)), rs.choice(Ml, len(Ml)))
            return pooled(a), pooled(b)
        return f
    def matched(rs):
        fs, ms, ws = [], [], []
        for t in tis_ok:
            g = meta_t[meta_t.tissue == t]; Ft, Mt = g.index[g.sex == 'female'].values, g.index[g.sex == 'male'].values
            if rs is not None: Ft, Mt = rs.choice(Ft, len(Ft)), rs.choice(Mt, len(Mt))
            fs.append(pooled(Ft)); ms.append(pooled(Mt)); ws.append(len(Ft) + len(Mt))
        av = lambda arr: np.ma.average(np.ma.masked_invalid(np.array(arr)), axis=0, weights=np.array(ws, float)).filled(np.nan)
        return av(fs), av(ms)
    views = {'all lines': (simple(Fall, Mall), len(Fall), len(Mall)),
             'non-sex-specific tissues': (simple(Fns, Mns), len(Fns), len(Mns))}
    if tis_ok:
        nF = int(sum(((meta_t.tissue == t) & (meta_t.sex == 'female')).sum() for t in tis_ok))
        nM = int(sum(((meta_t.tissue == t) & (meta_t.sex == 'male')).sum() for t in tis_ok))
        views[f'tissue-matched ({len(tis_ok)} tissues)'] = (matched, nF, nM)

    rng, rows = np.random.default_rng(seed), []
    for name, (fn, nF, nM) in views.items():
        if nF < 2 or nM < 2: continue
        f0, m0 = fn(None)
        boots = np.array([np.subtract(*fn(rng)) for _ in range(n_boot)])
        lo, hi = np.nanpercentile(boots, 2.5, axis=0), np.nanpercentile(boots, 97.5, axis=0)
        for k, mname in enumerate(METRICS):
            rows.append(dict(view=name, n_F_lines=nF, n_M_lines=nM, metric=mname, female=f0[k], male=m0[k],
                             gap_F_minus_M=f0[k] - m0[k], ci_lo=lo[k], ci_hi=hi[k]))
    res = pd.DataFrame(rows)

    # permutation test for the all-lines gap: shuffle sex labels WITHIN tissue (keeps tissue mix, tests a sex effect beyond it)
    obs = np.subtract(*views['all lines'][0](None)); null = []
    for _ in range(n_boot):
        sx = meta_t.sex.copy()
        for t, idx in meta_t.groupby('tissue').groups.items(): sx.loc[idx] = rng.permutation(meta_t.sex.loc[idx].values)
        null.append(np.subtract(pooled(meta_t.index[sx == 'female']), pooled(meta_t.index[sx == 'male'])))
    null = np.array(null)
    pv = [(1 + np.sum(np.abs(null[:, k][~np.isnan(null[:, k])]) >= abs(obs[k]))) / (1 + np.sum(~np.isnan(null[:, k]))) for k in range(3)]
    res['perm_p (all lines, sex shuffled within tissue)'] = np.where(res.view == 'all lines', res.metric.map(dict(zip(METRICS, pv))), np.nan)
    return res

def tissue_table(df, min_per_sex=MIN_PER_SEX):
    d = df[df.sex.isin(['female', 'male'])]; rows = []
    for t, g in d.groupby('tissue'):
        f, m = g[g.sex == 'female'], g[g.sex == 'male']
        if f.line.nunique() >= min_per_sex and m.line.nunique() >= min_per_sex:
            sf, sm = _stats(f.y, f.pred), _stats(m.y, m.pred)
            rows.append(dict(tissue=t, n_F=f.line.nunique(), n_M=m.line.nunique(),
                             **{f'{k} F': v for k, v in zip(METRICS, sf)}, **{f'{k} M': v for k, v in zip(METRICS, sm)},
                             **{f'gap {k}': a - b for k, a, b in zip(METRICS, sf, sm)}))
    return pd.DataFrame(rows)

def report(preds, model_name):
    """Prints and saves: performance, female-male gap (+CI, permutation p), within-tissue table, per-seed spread."""
    tag = f"{model_name.replace(' ', '_')}_{SPLIT_MODE}"
    first = preds[preds.seed == preds.seed.iloc[0]]
    perf, gap, tis = perf_table(first), gap_analysis(first), tissue_table(first)
    pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
    print(f"=== {model_name} | split: {SPLIT_MODE} | seed {first.seed.iloc[0]} ===\n")
    print('Performance on test pairs:'); print(perf.round(3).to_string(index=False))
    print('\nFemale-male gap (RMSE: positive = worse for women; Pearson/Spearman: negative = worse for women).')
    print('CI = 95% bootstrap over cell lines.'); 
    if gap is not None: print(gap.round(3).to_string(index=False))
    print('\nWithin-tissue comparison (tissues with >= %d lines of each sex in the test set):' % MIN_PER_SEX)
    print(tis.round(3).to_string(index=False) if len(tis) else '  none: not enough lines of both sexes per tissue in the test set')
    if preds.seed.nunique() > 1:
        per_seed = pd.DataFrame([perf_table(g).query("group == 'all'").iloc[0][METRICS] for _, g in preds.groupby('seed')])
        print('\nAcross seeds (mean +- std):'); print(pd.DataFrame({'mean': per_seed.mean(), 'std': per_seed.std()}).round(3))
    preds.assign(model=model_name, split_mode=SPLIT_MODE).to_csv(f'{OUT_DIR}/test_preds_{tag}.csv', index=False)
    perf.to_csv(f'{OUT_DIR}/perf_{tag}.csv', index=False)
    if gap is not None: gap.to_csv(f'{OUT_DIR}/gap_{tag}.csv', index=False)
    tis.to_csv(f'{OUT_DIR}/tissue_{tag}.csv', index=False)
    return perf, gap, tis

## 4. Train: Baseline1 expression-only

In [ ]:
OMICS = ['E']
H_DIMS = [128]
MODEL_NAME = 'Baseline1 expression-only'

In [ ]:
import copy, torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler
from utils import AllTripletSelector                      # from the MOLI repo
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
T = lambda a: torch.tensor(np.asarray(a), dtype=torch.float32, device=dev)
trip_sel = AllTripletSelector()
EPOCHS = 2 if SMOKE else 40

class MOLIReg(nn.Module):
    """One encoder per omic (Linear -> BatchNorm -> ReLU -> Dropout), concatenate, linear head -> ln IC50."""
    def __init__(self, in_dims, h_dims, rate=0.3):
        super().__init__()
        self.encs = nn.ModuleList([nn.Sequential(nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(rate))
                                   for d, h in zip(in_dims, h_dims)])
        self.head = nn.Linear(sum(h_dims), 1)
    def forward(self, xs):
        z = torch.cat([e(x) for e, x in zip(self.encs, xs)], dim=1)
        return self.head(z).squeeze(-1), z

# standardize expression / copy number with TRAINING-line statistics; mutations stay binary
feats = {}
for o in OMICS:
    X = A[o]
    if o in ('E', 'C'): X = StandardScaler().fit(X[tr_idx]).transform(X)
    feats[o] = T(X)
Xs = [feats[o] for o in OMICS]

def predict(model, li):
    model.eval()
    with torch.no_grad(): return model([x[torch.as_tensor(li, device=dev)] for x in Xs])[0].cpu().numpy()

def train_drug(d_tr, d_va, epochs, bs=32, lam=0.1, margin=1.0):
    model = MOLIReg([x.shape[1] for x in Xs], H_DIMS).to(dev)
    opt, mse, trip = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4), nn.MSELoss(), nn.TripletMarginLoss(margin=margin, p=2)
    y_tr = T(d_tr.y.values); lab = (y_tr < y_tr.median()).float()          # sensitive / resistant for the triplet loss
    bs = min(bs, max(4, len(d_tr) // 2)); best, best_state = np.inf, None
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(d_tr))
        for i in range(0, len(perm) - bs + 1, bs):
            b = perm[i:i + bs]
            pred, z = model([x[torch.as_tensor(d_tr.li.values[b.numpy()], device=dev)] for x in Xs])
            loss = mse(pred, y_tr[b])
            t = trip_sel.get_triplets(z, lab[b])
            if t.dim() == 2 and len(t) > 0: loss = loss + lam * trip(z[t[:, 0]], z[t[:, 1]], z[t[:, 2]])
            opt.zero_grad(); loss.backward(); opt.step()
        v = float(np.sqrt(np.mean((predict(model, d_va.li.values) - d_va.y.values) ** 2)))
        if v < best: best, best_state = v, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state); return model, best

all_preds = []
for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed); rows = []
    for cid in drugs:
        d = pairs[pairs.cid == cid]
        d_tr, d_va, d_te = d[d.split == 'train'], d[d.split == 'val'], d[d.split == 'test']
        if len(d_tr) < 20 or len(d_va) < 3 or len(d_te) < 3: print('skip', cid, '(too few lines)'); continue
        model, vr = train_drug(d_tr, d_va, EPOCHS)
        rows.append(d_te.assign(pred=predict(model, d_te.li.values)))
        print(f"seed {seed} drug {cid}: train={len(d_tr)} val RMSE={vr:.3f}")
    all_preds.append(pd.concat(rows).assign(seed=seed))
preds = pd.concat(all_preds, ignore_index=True)

## Results

In [ ]:
perf, gap, tis = report(preds, MODEL_NAME)